# Algorithm 4 — K-Nearest Neighbors (KNN)
**Member 4 | Telco Customer Churn Dataset**

---
### What is K-Nearest Neighbors?
KNN is one of the most intuitive machine learning algorithms. When it needs to classify a new customer, it looks at the **K most similar customers** (its 'nearest neighbors') in the training data and takes a vote from them.

**Example:** If K=5 and we have a new customer, the algorithm finds the 5 most similar customers already in the training data. If 4 of them churned, the model predicts this new customer will also churn.

**How 'similarity' is measured:**  
By default, KNN uses **Euclidean distance** — the straight-line distance between two data points in feature space.

**Why use it here?**  
- Very intuitive and easy to understand  
- No training phase required (it memorizes the data)  
- Good at capturing complex patterns  
- Requires **feature scaling** — very important!

---
## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)

print('All libraries imported successfully!')

---
## Step 2 — Load the Dataset

In [ ]:
df = pd.read_csv('Telco_csv.csv')

print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

---
## Step 3 — Explore the Dataset

In [ ]:
print('Dataset Info:')
df.info()

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())

In [ ]:
# Churn distribution
print('Churn value counts:')
print(df['Churn'].value_counts())

plt.figure(figsize=(5, 4))
df['Churn'].value_counts().plot(kind='bar', color=['mediumpurple', 'tomato'], edgecolor='black')
plt.title('Churn Distribution')
plt.xlabel('Churn')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('knn_churn_distribution.png', dpi=150)
plt.show()
print('Saved: knn_churn_distribution.png')

In [ ]:
# Tenure vs Monthly Charges — scatter colored by churn
plt.figure(figsize=(7, 5))
colors = df['Churn'].map({'Yes': 'tomato', 'No': 'steelblue'})
plt.scatter(df['tenure'], df['MonthlyCharges'], c=colors, alpha=0.3, s=15)
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='tomato', label='Churn'),
                   Patch(facecolor='steelblue', label='No Churn')]
plt.legend(handles=legend_elements)
plt.xlabel('Tenure (months)')
plt.ylabel('Monthly Charges ($)')
plt.title('Tenure vs Monthly Charges by Churn', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('knn_scatter_exploration.png', dpi=150)
plt.show()
print('Saved: knn_scatter_exploration.png')

---
## Step 4 — Data Preprocessing

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)
print('Shape after cleaning:', df.shape)

In [ ]:
df.drop('customerID', axis=1, inplace=True)

le = LabelEncoder()
df['Churn'] = le.fit_transform(df['Churn'])

binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

df['gender'] = le.fit_transform(df['gender'])

multi_cols = [
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaymentMethod'
]
df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

print('Final dataset shape after encoding:', df.shape)

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {len(X_train)}')
print(f'Testing samples  : {len(X_test)}')

In [ ]:
# IMPORTANT: KNN is very sensitive to feature scale.
# Without scaling, features with large ranges (like TotalCharges in the thousands)
# will dominate over features with small ranges (like SeniorCitizen = 0 or 1).

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Feature scaling done — this is essential for KNN!')

---
## Step 5 — Find the Best K Value

In [ ]:
# Try K values from 1 to 30 and find the one with the best test accuracy
k_values = range(1, 31)
train_accs, test_accs = [], []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_accs.append(accuracy_score(y_train, knn.predict(X_train_scaled)))
    test_accs.append(accuracy_score(y_test, knn.predict(X_test_scaled)))

best_k = k_values[np.argmax(test_accs)]
best_acc = max(test_accs)

print(f'Best K value  : {best_k}')
print(f'Best Accuracy : {best_acc*100:.2f}%')

# Plot
plt.figure(figsize=(9, 5))
plt.plot(k_values, train_accs, 'o-', color='steelblue', label='Training Accuracy')
plt.plot(k_values, test_accs,  's-', color='mediumpurple', label='Test Accuracy')
plt.axvline(best_k, color='black', linestyle='--', alpha=0.7, label=f'Best K = {best_k}')
plt.xlabel('K (Number of Neighbors)')
plt.ylabel('Accuracy')
plt.title('KNN — Finding the Best K Value', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('knn_best_k.png', dpi=150)
plt.show()
print('Saved: knn_best_k.png')

---
## Step 6 — Train the KNN Model with Best K

In [ ]:
model_knn = KNeighborsClassifier(n_neighbors=best_k)
model_knn.fit(X_train_scaled, y_train)

print(f'Model training complete with K = {best_k}!')

---
## Step 7 — Evaluate the Model

In [ ]:
y_pred_knn = model_knn.predict(X_test_scaled)
y_prob_knn = model_knn.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test, y_pred_knn)
roc_auc  = roc_auc_score(y_test, y_prob_knn)

print('=' * 45)
print('           KNN RESULTS')
print('=' * 45)
print(f'  K Value   : {best_k}')
print(f'  Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'  ROC-AUC   : {roc_auc:.4f}')
print('=' * 45)
print('\nClassification Report:')
print(classification_report(y_test, y_pred_knn, target_names=['No Churn', 'Churn']))

In [ ]:
# Cross-validation
cv_scores = cross_val_score(model_knn, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f'Cross-Validation Accuracy (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Individual fold scores: {[round(s,4) for s in cv_scores]}')

---
## Step 8 — Visualizations

In [ ]:
# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred_knn)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
plt.title(f'KNN (K={best_k}) — Confusion Matrix', fontsize=13, fontweight='bold')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('knn_confusion_matrix.png', dpi=150)
plt.show()
print('Saved: knn_confusion_matrix.png')

In [ ]:
# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_knn)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='mediumpurple', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'KNN (K={best_k}) — ROC Curve', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('knn_roc_curve.png', dpi=150)
plt.show()
print('Saved: knn_roc_curve.png')

---
## Step 9 — Summary

In [ ]:
print('=' * 50)
print('     K-NEAREST NEIGHBORS — FINAL SUMMARY')
print('=' * 50)
print(f'  Algorithm         : K-Nearest Neighbors (KNN)')
print(f'  Dataset           : Telco Customer Churn')
print(f'  Total Samples     : {len(df)}')
print(f'  Training Samples  : {len(X_train)}')
print(f'  Testing Samples   : {len(X_test)}')
print(f'  Features Used     : {X.shape[1]}')
print(f'  Best K Value      : {best_k}')
print(f'  Test Accuracy     : {accuracy*100:.2f}%')
print(f'  ROC-AUC Score     : {roc_auc:.4f}')
print(f'  CV Accuracy (5-fold): {cv_scores.mean()*100:.2f}%')
print('=' * 50)